# ClusterFinder — CPU vs CUDA

Three ways to find clusters in the same frames, from the same pedestal:

| | |
|---|---|
| **`ClusterFinderMT`** + `ClusterCollector` | multithreaded CPU baseline |
| **`ClusterFinderCUDA.find_clusters_batched`** | batched across CUDA streams, over a pinned input buffer, so H2D / kernel / D2H overlap |
| **`find_cluster_views_batched_iter`** | the same GPU pipeline, zero-copy: reductions such as a spectrum are read straight from the GPU output, no clusters are copied out |

**Part 1** times them. **Part 2** compares what the CPU and the GPU found.

In [ ]:
import sys; sys.path.append('/home/ferjao_k/aare/build')

import time
from pathlib import Path

import boost_histogram as bh
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from aare import (File, ClusterFinderMT, ClusterCollector, ClusterFinderCUDA,
                  find_cluster_views_batched_iter)

## Configuration

In [ ]:
base = Path('/mnt/sls_det_storage/moench_data/2603_MaxIVBeamtime/2026032408/process/xrf/')
f = File(base / 'Cu_factor_10_data_master_0.json')
pd = File(base / 'Cu_factor_10_pedestal_master_0.json')

n_frames_pd  = 1000                  # pedestal frames, taken from the head of the file
N            = 100000                 # data frames, read after those
cluster_size = (3, 3)
rows, cols   = f.rows, f.cols
image_size   = (rows, cols)

N_SIGMA    = 5
N_STREAMS  = 4                       # H2D / kernel / D2H overlap
BATCH_SIZE = 3000                    # frames per find_clusters_batched call

# CPU workers. ClusterFinderMT gives each thread its own ClusterFinder with its
# own pedestal, and data frames are handed out round-robin -- so at high thread
# counts each pedestal tracks on only 1/n of the frames while the CUDA finder's
# single pedestal sees all of them, and the cluster counts drift apart. Keep it
# low when comparing counts; raise it to measure CPU throughput.
N_THREADS = 24

# Fixed per-frame output slot the kernel writes into, copied WHOLE on every D2H.
# Too low does not error -- it TRUNCATES. The CUDA cell prints the peak actually
# reached, which is how you size this for a new dataset.
CAP = 3000

capacity = 3_000                    # initial ClusterVector capacity, CPU side
N_BINS   = 200
E_MIN, E_MAX = -2, 4000              # spectrum range, ADU

print(f'Image size:      {image_size}   cluster {cluster_size}')
print(f'Pedestal frames: {n_frames_pd}')
print(f'Data frames:     {N}  of {f.total_frames} in file')

## Build the finders

In [ ]:
cf_cpu = ClusterFinderMT(image_size, #only positional argument here i.e. need to use the keyword for next ones
                         cluster_size=cluster_size, 
                         n_sigma=N_SIGMA,
                         capacity=capacity, 
                         min_pedestal_samples=n_frames_pd, 
                         n_threads=N_THREADS)

sink = ClusterCollector(cf_cpu)

cf_cuda = ClusterFinderCUDA(image_size, 
                            cluster_size=cluster_size, 
                            n_sigma=N_SIGMA,
                            max_clusters_per_frame=CAP, 
                            n_streams=N_STREAMS, 
                            min_pedestal_samples=n_frames_pd, 
                            algorithm="fixed_window", 
                            time_kernels=False)

# Same settings, for section 3. A finder's pedestal keeps tracking while it
# runs, so a second one fed the same pedestal frames starts where cf_cuda did.
cf_view = ClusterFinderCUDA(image_size,
                            cluster_size=cluster_size,
                            n_sigma=N_SIGMA,
                            max_clusters_per_frame=CAP,
                            n_streams=N_STREAMS,
                            min_pedestal_samples=n_frames_pd,
                            algorithm="fixed_window")

## Pedestal

Identical frames into all three, so the implementation is the only thing that differs.

In [ ]:
pd.seek(0)
t0 = time.perf_counter()
for _ in range(n_frames_pd):
    pd_img = pd.read_frame()
    cf_cpu.push_pedestal_frame(pd_img.copy())
    cf_cuda.push_pedestal_frame(pd_img.copy())
    cf_view.push_pedestal_frame(pd_img.copy())
print(f'Pedestal ({n_frames_pd} frames): {time.perf_counter() - t0:.2f}s')

## Read the data frames

Kept out of every timing loop: all finders run over the same in-memory array.

In [ ]:
f.seek(0)
t0 = time.perf_counter()
data = f.read_n(N)
t_io = time.perf_counter() - t0
gb = f.bytes_per_frame * N / 1e9
print(f'Reading {N} frames: {t_io:.2f}s  ({N / t_io:,.0f} FPS, '
      f'{gb:.2f} GB at {gb / t_io:.2f} GB/s)')

## 1 — CPU, multithreaded

In [ ]:
t0 = time.perf_counter()
# Pass the frame number: find_clusters(frame) defaults it to 0, and every
# resulting cluster is then stamped frame 0. Part 2 groups by it.
for i, frame in enumerate(tqdm(data)):
    cf_cpu.find_clusters(frame, i)
t_cpu = time.perf_counter() - t0

cf_cpu.stop(); sink.stop()
clusters_cpu = sink.steal_clusters()

t0 = time.perf_counter()
hist_cpu = bh.Histogram(bh.axis.Regular(N_BINS, E_MIN, E_MAX))
n_clusters_cpu = 0
for cv in clusters_cpu:
    hist_cpu.fill(cv.sum())
    n_clusters_cpu += cv.size
t_hist_cpu = time.perf_counter() - t0

print(f'CPU MT ({N_THREADS} threads): {t_cpu:.2f}s   {N / t_cpu:,.0f} FPS   '
      f'{n_clusters_cpu:,} clusters ({n_clusters_cpu / N:.1f}/frame)')
print(f'  spectrum filled in {t_hist_cpu:.2f}s')

## 2 — CUDA, batched + multi-streamed

`register_input_buffer` page-locks the input once so H2D runs at full DMA
bandwidth instead of through the driver's staging buffer. It is paid once,
outside the loop, and released when the dataset is done.

In [ ]:
t0 = time.perf_counter()
cf_cuda.register_input_buffer(data)
t_pin = time.perf_counter() - t0

clusters_cuda = []
t0 = time.perf_counter()
for start in range(0, N, BATCH_SIZE):
    stop = min(start + BATCH_SIZE, N)
    clusters_cuda.extend(
        cf_cuda.find_clusters_batched(data[start:stop], first_frame=start))
t_cuda = time.perf_counter() - t0

cf_cuda.unregister_input_buffer()

t0 = time.perf_counter()
hist_cuda = bh.Histogram(bh.axis.Regular(N_BINS, E_MIN, E_MAX))
energies = [np.asarray(cv.sum()).ravel() for cv in clusters_cuda if cv.size > 0]
if energies:
    hist_cuda.fill(np.concatenate(energies))
t_hist_cuda = time.perf_counter() - t0
n_clusters_cuda = sum(cv.size for cv in clusters_cuda)

print(f'CUDA batched: {t_cuda:.2f}s   {N / t_cuda:,.0f} FPS   '
      f'{n_clusters_cuda:,} clusters ({n_clusters_cuda / N:.1f}/frame)')
print(f'  spectrum filled in {t_hist_cuda:.2f}s')
print(f'  input pinned in {t_pin:.2f}s, once, outside the loop')
print(f'  vs CPU: {t_cpu / t_cuda:.2f}x')

# One ClusterVector per frame, so this IS the per-frame maximum.
peak = max((cv.size for cv in clusters_cuda), default=0)
print(f'  peak clusters in one frame: {peak:,} of CAP {CAP:,}')
if peak >= CAP:
    print('  !! CAP REACHED -- clusters were dropped; raise CAP and re-run.')

## 3 — CUDA, zero-copy views

`find_cluster_views_batched_iter` drives the same GPU pipeline as section 2,
but copies nothing out. Each `BatchView` points into the finder's pinned output
buffer and is valid only inside the loop body. Its reductions return owned
arrays: `v.sums()` the energy of every cluster in the chunk, `v.counts` the
number of clusters per frame. A spectrum is filled from them directly, without
a `ClusterVector` per frame.

Use section 2 when you need the clusters themselves, to save or analyse them.
Use this when a reduction is all you need, for example to re-render a spectrum
quickly with another binning or range.

`cf_view` got the same pedestal frames as `cf_cuda`, so it finds exactly the
same clusters. A host buffer can be pinned by one finder at a time, which is
why section 2 released `data` when it was done.

A second pass only counts the clusters, to time the pipeline without the
spectrum. Its pedestal has kept tracking through the first pass, so its
counts differ slightly; only its time is used.

In [ ]:
cf_view.register_input_buffer(data)
# The iterator works in larger chunks than section 2. Page-lock its two output
# slots now, so that one-off cost is not charged to the loop.
cf_view.reserve_output_slots(cf_view.chunk_size_for(N))

hist_view = bh.Histogram(bh.axis.Regular(N_BINS, E_MIN, E_MAX))
n_clusters_view = peak_view = 0
t0 = time.perf_counter()
for v in find_cluster_views_batched_iter(cf_view, data):
    hist_view.fill(v.sums())       # one array per chunk, nothing per frame
    counts = v.counts              # clusters in each frame of the chunk
    n_clusters_view += int(counts.sum())
    peak_view = max(peak_view, int(counts.max()))
t_view = time.perf_counter() - t0

# The pipeline alone, for the summary: read one counter per chunk.
t0 = time.perf_counter()
for v in find_cluster_views_batched_iter(cf_view, data):
    v.total_clusters
t_view_find = time.perf_counter() - t0

cf_view.unregister_input_buffer()

print(f'CUDA zero-copy: {t_view:.2f}s   {N / t_view:,.0f} FPS   '
      f'{n_clusters_view:,} clusters, spectrum included')
print(f'  pipeline alone: {t_view_find:.2f}s   {N / t_view_find:,.0f} FPS')
print(f'  same spectrum as section 2: '
      f'{np.array_equal(hist_view.values(), hist_cuda.values())}')
print(f'  peak clusters in one frame: {peak_view:,} of CAP {CAP:,}')
if peak_view >= CAP:
    print('  !! CAP REACHED -- clusters were dropped; raise CAP and re-run.')

## Summary

In [ ]:
# For each finder: the pipeline alone (find), and the time to a filled
# spectrum (total). "spectrum" is what the histogram adds: total - find.
rows_sum = (('CPU MT (%d thr)' % N_THREADS, t_cpu, t_cpu + t_hist_cpu, n_clusters_cpu),
            ('CUDA batched', t_cuda, t_cuda + t_hist_cuda, n_clusters_cuda),
            ('CUDA zero-copy', t_view_find, t_view, n_clusters_view))

print(f'{"":<18} {"find s":>7} {"find FPS":>9} {"spectrum s":>11} '
      f'{"total s":>8} {"FPS with spectrum":>18} {"clusters":>13}')
for name, t_find, t_total, n in rows_sum:
    print(f'{name:<18} {t_find:7.2f} {N / t_find:9,.0f} {t_total - t_find:11.2f} '
          f'{t_total:8.2f} {N / t_total:18,.0f} {n:>13,}')

print(f'\nspeedup over the CPU, finding:          '
      f'batched {t_cpu / t_cuda:.1f}x, zero-copy {t_cpu / t_view_find:.1f}x')
print(f'speedup over the CPU, to a spectrum:    '
      f'batched {(t_cpu + t_hist_cpu) / (t_cuda + t_hist_cuda):.1f}x, '
      f'zero-copy {(t_cpu + t_hist_cpu) / t_view:.1f}x')

# Part 2 — Where the two disagree

The two implementations are not bit-identical by construction: the CUDA finder
updates its pedestal once per frame, the CPU finder per pixel, so once the
pedestals drift apart a few pixels land on opposite sides of the threshold.

Three views, cheapest first: the **spectra**, the **cluster sets**, and **where
in the run** the difference accumulates.

## Spectra

Both curves should lie on top of each other; the ratio panel is the sensitive
view.

In [ ]:
fig, (ax_spec, ax_ratio) = plt.subplots(
    2, 1, figsize=(9, 6.5), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

edges = hist_cpu.axes[0].edges
cpu_vals, cuda_vals = hist_cpu.values(), hist_cuda.values()

ax_spec.stairs(cpu_vals, edges, label=f'CPU MT  ({n_clusters_cpu:,})',
               color='C0', linewidth=1.4)
ax_spec.stairs(cuda_vals, edges, label=f'CUDA  ({n_clusters_cuda:,})',
               color='C1', linestyle='--', linewidth=1.4)
ax_spec.set_ylabel('Counts')
ax_spec.set_yscale('log')
ax_spec.set_ylim(bottom=10)
ax_spec.set_title(f'Cluster energy spectrum — {N:,} frames')
ax_spec.legend()
ax_spec.grid(alpha=0.2)

with np.errstate(divide='ignore', invalid='ignore'):
    ratio = np.where(cpu_vals > 0, cuda_vals / cpu_vals, np.nan)
ax_ratio.stairs(ratio, edges, color='C1', linewidth=1.2)
ax_ratio.axhline(1.0, color='gray', linewidth=0.5)
ax_ratio.set_ylabel('CUDA / CPU')
ax_ratio.set_xlabel('Energy [ADU]')
ax_ratio.set_ylim(0.9, 1.1)
ax_ratio.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Cluster sets

Every cluster is keyed by `(frame, x, y)`, so the two results can be differenced
directly — no per-frame Python loop.

A cluster whose centre moved by one pixel was *relabelled*, not gained or lost,
so those are counted separately from genuine adds and drops.

In [ ]:
def cluster_keys(vectors):
    """One int64 key per cluster: frame * rows*cols + y * cols + x.

    Centres are unique within a frame, so keys are globally unique and the sets
    below need only a sort, never a np.unique.
    """
    n = len(vectors)
    sizes = np.fromiter((cv.size for cv in vectors), np.int64, n)
    frames = np.repeat(
        np.fromiter((cv.frame_number for cv in vectors), np.int64, n), sizes)
    xy = [np.asarray(cv) for cv in vectors if cv.size]
    if not xy:
        return np.empty(0, np.int64), frames
    x = np.concatenate([v['x'] for v in xy]).astype(np.int64)
    y = np.concatenate([v['y'] for v in xy]).astype(np.int64)
    return frames * (rows * cols) + y * cols + x, frames


t0 = time.perf_counter()
k_cpu, fr_cpu = cluster_keys(clusters_cpu)
k_cuda, fr_cuda = cluster_keys(clusters_cuda)

# Sort ONCE. np.setdiff1d and np.isin each re-sort their argument on every call,
# and the shift check below calls one eight times per direction -- about twenty
# sorts of a 27-million-element array. Two sorts plus binary search instead.
s_cpu = np.sort(k_cpu)
s_cuda = np.sort(k_cuda)


def present_in(vals, sorted_keys):
    """Which of `vals` appear in an already-sorted key array."""
    if sorted_keys.size == 0:
        return np.zeros(vals.size, dtype=bool)
    i = np.searchsorted(sorted_keys, vals)
    np.clip(i, 0, sorted_keys.size - 1, out=i)
    return sorted_keys[i] == vals


cpu_only = s_cpu[~present_in(s_cpu, s_cuda)]
cuda_only = s_cuda[~present_in(s_cuda, s_cpu)]


def drop_shifted(only, sorted_other):
    """Drop mismatches that have a counterpart within 1 px in the other set."""
    keep = np.ones(only.size, dtype=bool)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dx or dy:
                keep &= ~present_in(only + dy * cols + dx, sorted_other)
    return only[keep]


cpu_real = drop_shifted(cpu_only, s_cuda)
cuda_real = drop_shifted(cuda_only, s_cpu)
touched = np.unique(np.concatenate([cpu_real, cuda_real]) // (rows * cols)).size

print(f'CPU  {k_cpu.size:>12,} clusters')
print(f'CUDA {k_cuda.size:>12,} clusters      net {k_cuda.size - k_cpu.size:+,}'
      f'  ({abs(k_cuda.size - k_cpu.size) / k_cpu.size:.4%})')
print()
print(f'exact-position mismatches   CPU-only {cpu_only.size:>8,}   '
      f'CUDA-only {cuda_only.size:>8,}')
print(f'  of which shifted by 1 px  {cpu_only.size - cpu_real.size:>17,}   '
      f'{cuda_only.size - cuda_real.size:>17,}')
print(f'  genuine add / drop        {cpu_real.size:>17,}   {cuda_real.size:>17,}')
print(f'\nframes touched by a genuine difference: {touched:,} of {N:,}')
print(f'[{time.perf_counter() - t0:.1f}s]')

## Where in the run it accumulates

Cumulative `CUDA − CPU` against frame index. A curve that rises and then flattens
means the two pedestals settled into agreement and the difference is a start-up
transient; a steadily sloping line would mean a persistent per-frame bias.

In [ ]:
cnt_cpu = np.bincount(fr_cpu, minlength=N)
cnt_cuda = np.bincount(fr_cuda, minlength=N)
cum = np.cumsum(cnt_cuda.astype(np.int64) - cnt_cpu.astype(np.int64))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(cum, color='C1', linewidth=1.2)
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Frame')
ax.set_ylabel('cumulative CUDA - CPU')
ax.set_title(f'Cluster-count difference accumulating over {N:,} frames '
             f'(final {cum[-1]:+,})')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

half = N // 2
print(f'first half: {cum[half - 1]:+,}      second half: {cum[-1] - cum[half - 1]:+,}')

# Appendix — host memory and page faults

Everything above is the cluster finder. This appendix is about the host memory
that `find_clusters_batched` fills, and does not change any result.

Each frame's clusters go into a new `ClusterVector`. The first write to a page
the process has never used is a *minor page fault*: the kernel maps the page
and zero-fills it. Two things change how much that costs:

- **glibc's heap settings.** By default glibc grows its heap a little at a
  time, so the kernel backs it with 4 KiB pages, and it gives freed memory back,
  so the next batch faults it in again. `GLIBC_TUNABLES` (read once, at process
  start) can make it grow the heap 256 MiB at a time, which the kernel can back
  with huge pages when memory is free, and keep freed memory, so later batches
  reuse pages that are already mapped.
- **Transparent huge pages.** When they back the heap, one fault maps and
  zero-fills 2 MiB instead of 4 KiB. The fault count drops by up to 512x, but
  the zero-filling remains: compare the times, not only the counts.

**Pre-heating** grows the heap and writes every page *before* the timed loop,
then frees it; with glibc told to keep freed memory, the loop then reuses those
pages. It only moves the zero-filling out of the loop, which is what a
long-running acquisition wants, paid once at start-up.

To run this notebook with the setting, register a Jupyter kernel that carries
it, then pick "Python (heap kept)" as the notebook's kernel in VS Code or
Jupyter (`jupyter kernelspec uninstall py-heap` removes it):

```bash
python -m ipykernel install --user --name py-heap \
    --display-name "Python (heap kept)" \
    --env GLIBC_TUNABLES glibc.malloc.mmap_threshold=33554432:glibc.malloc.trim_threshold=4294967295:glibc.malloc.top_pad=268435456
```

The documentation page for `ClusterFinderCUDA` ("Performance") explains each
part of the setting and shows how to pre-heat.

The cell below shows this kernel's setting and how much of its memory sits in
huge pages. To compare the setting, pre-heating and the zero-copy path, run
`python/tests/perf/cf/page_faults.py` from a terminal, once per configuration,
with no notebook kernel holding memory: a kernel that has run the cells above
holds tens of GB, which leaves too little free memory for huge pages and
cannot show a cold start. Its README gives the commands and reference numbers.

In [ ]:
import os

thp = Path('/sys/kernel/mm/transparent_hugepage/enabled').read_text().split()
smaps = {k: v for k, v in (line.split(':', 1) for line in
         Path('/proc/self/smaps_rollup').read_text().splitlines()[1:])}
gib = lambda field: int(smaps[field].split()[0]) / 2**20

print('This kernel')
print(f"  GLIBC_TUNABLES          {os.environ.get('GLIBC_TUNABLES', 'not set')}")
print(f"  transparent huge pages  {next(w.strip('[]') for w in thp if w.startswith('['))}")
print(f'  resident                {gib("Rss"):.1f} GiB, '
      f'{gib("AnonHugePages"):.1f} GiB of it in huge pages')
print(f'  Part 1, CUDA loop       {N / t_cuda:,.0f} FPS over {N:,} frames')